# Step 5: Decision Engine
## Menghasilkan Keputusan Otomatis Platform

**Tujuan**:
- Baca hasil Scoring Engine dari `product_scores.csv`
- Terapkan threshold rules untuk 3 kategori keputusan
- Hasilkan kolom decision & action untuk setiap produk
- Output: `product_decisions.csv` (file output final sistem)

---

## Threshold Rules

| Kondisi | Keputusan | Aksi Platform |
|---------|-----------|---------------|
| score ≥ 0.65 | PERTAHANKAN | Listing tetap aktif, tidak ada tindakan |
| 0.35 ≤ score < 0.65 | EVALUASI | Notifikasi seller: perbaiki aspek [weakness_flag] |
| score < 0.35 | TARIK | Listing di-suspend, review ulang 30 hari |

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set up paths
BASE_PATH = Path('../')
PRODUCT_SCORES = BASE_PATH / 'product_scores.csv'
OUTPUT_PATH = BASE_PATH / 'product_decisions.csv'

print(f"Checking files...")
print(f"Product Scores exists: {PRODUCT_SCORES.exists()}")

Checking files...
Product Scores exists: True


## 1. Load Product Scores

In [2]:
# Load product scores dari Step 4
df_decisions = pd.read_csv(PRODUCT_SCORES)

print(f"Shape: {df_decisions.shape}")
print(f"\nColumns: {df_decisions.columns.tolist()}")
print(f"\nFirst 5 rows:")
print(df_decisions.head())

print(f"\nProduct Score Statistics:")
print(df_decisions['product_score'].describe())

Shape: (14948, 8)

Columns: ['product_id', 'review_count', 'aspect_harga_score', 'aspect_kualitas_score', 'aspect_pengiriman_score', 'aspect_layanan_score', 'product_score', 'weakness_flags']

First 5 rows:
   product_id  review_count  aspect_harga_score  aspect_kualitas_score  \
0  B007B5TRQ0             1                 1.0                    1.0   
1  B0007U0IGE             1                 1.0                    1.0   
2  B0040Y8YP0             1                 1.0                    1.0   
3  B00LIQ3HWS             1                 1.0                    1.0   
4  B00E7MAL6U             1                 1.0                    1.0   

   aspect_pengiriman_score  aspect_layanan_score  product_score weakness_flags  
0                      1.0                   1.0            1.0           NONE  
1                      1.0                   1.0            1.0           NONE  
2                      1.0                   1.0            1.0           NONE  
3                      1

## 2. Apply Decision Threshold

In [3]:
# Define thresholds
THRESHOLD_KEEP = 0.65       # score >= 0.65 = PERTAHANKAN
THRESHOLD_EVALUATE = 0.35   # 0.35 <= score < 0.65 = EVALUASI
# score < 0.35 = TARIK

def assign_decision(score):
    """
    Assign decision based on product_score threshold.
    """
    if score >= THRESHOLD_KEEP:
        return 'PERTAHANKAN'
    elif score >= THRESHOLD_EVALUATE:
        return 'EVALUASI'
    else:
        return 'TARIK'

# Apply decision logic
df_decisions['decision'] = df_decisions['product_score'].apply(assign_decision)

print("Decision Distribution:")
print(df_decisions['decision'].value_counts())
print(f"\nPercentage:")
print((df_decisions['decision'].value_counts(normalize=True) * 100).round(2))

Decision Distribution:
decision
EVALUASI       10557
TARIK           2399
PERTAHANKAN     1992
Name: count, dtype: int64

Percentage:
decision
EVALUASI       70.62
TARIK          16.05
PERTAHANKAN    13.33
Name: proportion, dtype: float64


## 3. Generate Action Items

In [4]:
def generate_action(row):
    """
    Generate platform action based on decision.
    """
    decision = row['decision']
    weakness = row['weakness_flags']
    
    if decision == 'PERTAHANKAN':
        return 'Listing tetap aktif. Tidak ada tindakan.'
    elif decision == 'EVALUASI':
        return f'NOTIFIKASI SELLER: Perbaiki aspek [{weakness}]. Target ulang 14 hari.'
    else:  # TARIK
        return 'Listing di-suspend. Review ulang 30 hari.'

# Apply action generation
df_decisions['action'] = df_decisions.apply(generate_action, axis=1)

print("Sample Actions:")
print(df_decisions[['product_id', 'product_score', 'decision', 'weakness_flags', 'action']].head(15))

Sample Actions:
    product_id  product_score     decision weakness_flags  \
0   B007B5TRQ0            1.0  PERTAHANKAN           NONE   
1   B0007U0IGE            1.0  PERTAHANKAN           NONE   
2   B0040Y8YP0            1.0  PERTAHANKAN           NONE   
3   B00LIQ3HWS            1.0  PERTAHANKAN           NONE   
4   B00E7MAL6U            1.0  PERTAHANKAN           NONE   
5   B00J0BXPD4            1.0  PERTAHANKAN           NONE   
6   B00LHYX4UQ            1.0  PERTAHANKAN           NONE   
7   B00MMNZEHW            1.0  PERTAHANKAN           NONE   
8   B0009RMGK6            1.0  PERTAHANKAN           NONE   
9   B00GC58ESW            1.0  PERTAHANKAN           NONE   
10  B002TLUJB8            1.0  PERTAHANKAN           NONE   
11  B003SOQN92            1.0  PERTAHANKAN           NONE   
12  B003WSKKAM            1.0  PERTAHANKAN           NONE   
13  B001MSXDHQ            1.0  PERTAHANKAN           NONE   
14  B004X086LS            1.0  PERTAHANKAN           NONE   

       

## 4. Assign Priority Level

In [5]:
def assign_priority(row):
    """
    Assign priority based on decision and weakness count.
    """
    decision = row['decision']
    weakness = row['weakness_flags']
    score = row['product_score']
    
    if decision == 'TARIK':
        if score < 0.25:
            return 'CRITICAL'
        else:
            return 'HIGH'
    elif decision == 'EVALUASI':
        weakness_count = len(weakness.split(',')) if weakness != 'NONE' else 0
        return 'MEDIUM' if weakness_count < 2 else 'HIGH'
    else:  # PERTAHANKAN
        return 'LOW'

df_decisions['priority'] = df_decisions.apply(assign_priority, axis=1)

print("Priority Distribution:")
print(df_decisions['priority'].value_counts())
print(f"\nCritical Items:")
print(df_decisions[df_decisions['priority'] == 'CRITICAL'].shape[0])

Priority Distribution:
priority
MEDIUM      9821
HIGH        2428
LOW         1992
CRITICAL     707
Name: count, dtype: int64

Critical Items:
707


## 5. Add Timeline

In [6]:
today = datetime.now().date()

def assign_deadline(row):
    decision = row['decision']
    if decision == 'PERTAHANKAN':
        return None  # No deadline
    elif decision == 'EVALUASI':
        deadline = today + timedelta(days=14)
        return deadline
    else:  # TARIK
        deadline = today + timedelta(days=30)
        return deadline

df_decisions['deadline'] = df_decisions.apply(assign_deadline, axis=1)
df_decisions['created_date'] = today

print(f"Deadlines assigned:")
print(df_decisions[df_decisions['deadline'].notna()][['product_id', 'decision', 'deadline']].head(10))

Deadlines assigned:
      product_id  decision    deadline
1992  B000TB0RY4  EVALUASI  2026-06-20
1993  B004SCSV2U  EVALUASI  2026-06-20
1994  B008V9RW58  EVALUASI  2026-06-20
1995  B007SP2CO2  EVALUASI  2026-06-20
1996  B000I3F91O  EVALUASI  2026-06-20
1997  B004OBZ2XQ  EVALUASI  2026-06-20
1998  B00BEWF4R2  EVALUASI  2026-06-20
1999  B00005AW1E  EVALUASI  2026-06-20
2000  B003ANZF0E  EVALUASI  2026-06-20
2001  B002M3SOME  EVALUASI  2026-06-20


## 6. Final Output

In [7]:
# Select final output columns
output_cols = [
    'product_id',
    'review_count',
    'aspect_harga_score',
    'aspect_kualitas_score',
    'aspect_pengiriman_score',
    'aspect_layanan_score',
    'product_score',
    'weakness_flags',
    'decision',
    'priority',
    'action',
    'deadline',
    'created_date'
]

df_final = df_decisions[output_cols].copy()

# Sort by priority and score
priority_order = {'CRITICAL': 0, 'HIGH': 1, 'MEDIUM': 2, 'LOW': 3}
df_final['priority_num'] = df_final['priority'].map(priority_order)
df_final = df_final.sort_values(['priority_num', 'product_score'], ascending=[True, False]).drop('priority_num', axis=1)

print(f"\nFinal Output Shape: {df_final.shape}")
print(f"\n CRITICAL & HIGH Priority Items:")
print(df_final[df_final['priority'].isin(['CRITICAL', 'HIGH'])][['product_id', 'product_score', 'decision', 'priority', 'weakness_flags']].head(20))

print(f"\n Sample PERTAHANKAN Products:")
print(df_final[df_final['decision'] == 'PERTAHANKAN'][['product_id', 'product_score', 'decision', 'weakness_flags']].head(5))


Final Output Shape: (14948, 13)

 CRITICAL & HIGH Priority Items:
       product_id  product_score decision  priority  \
14241  B007LOWIQW       0.250000    TARIK  CRITICAL   
14242  B000EPHR0C       0.250000    TARIK  CRITICAL   
14243  B003JHJL38       0.250000    TARIK  CRITICAL   
14244  B002Y0UB3K       0.250000    TARIK  CRITICAL   
14245  B001GT185K       0.234375    TARIK  CRITICAL   
14246  B000V5L5MG       0.229167    TARIK  CRITICAL   
14247  B002NEGTOC       0.229167    TARIK  CRITICAL   
14248  B004AKN53K       0.218750    TARIK  CRITICAL   
14249  B0000CEORU       0.214286    TARIK  CRITICAL   
14250  B004HD4L2E       0.208333    TARIK  CRITICAL   
14251  B001VEI290       0.208333    TARIK  CRITICAL   
14252  B003VGUGBS       0.208333    TARIK  CRITICAL   
14253  B000S5UY2G       0.208333    TARIK  CRITICAL   
14254  B000XPG2YU       0.208333    TARIK  CRITICAL   
14255  B004TS2AOS       0.208333    TARIK  CRITICAL   
14256  B004CLYJ1O       0.187500    TARIK  CRITICAL  

In [8]:
# Save to CSV
df_final.to_csv(OUTPUT_PATH, index=False)
print(f" Saved to: {OUTPUT_PATH}")
print(f"File size: {OUTPUT_PATH.stat().st_size / 1024:.2f} KB")
print(f"\n Summary:")
print(f"  - Total Products: {len(df_final)}")
print(f"  - PERTAHANKAN: {(df_final['decision'] == 'PERTAHANKAN').sum()} ({(df_final['decision'] == 'PERTAHANKAN').sum() / len(df_final) * 100:.1f}%)")
print(f"  - EVALUASI: {(df_final['decision'] == 'EVALUASI').sum()} ({(df_final['decision'] == 'EVALUASI').sum() / len(df_final) * 100:.1f}%)")
print(f"  - TARIK: {(df_final['decision'] == 'TARIK').sum()} ({(df_final['decision'] == 'TARIK').sum() / len(df_final) * 100:.1f}%)")

 Saved to: ..\product_decisions.csv
File size: 2108.36 KB

 Summary:
  - Total Products: 14948
  - PERTAHANKAN: 1992 (13.3%)
  - EVALUASI: 10557 (70.6%)
  - TARIK: 2399 (16.0%)


## 7. Decision Analysis

In [11]:
print("DECISION ENGINE - ANALYSIS REPORT")

print(f"\n Decision Summary:")
print(df_final['decision'].value_counts())

print(f"\n Priority Distribution:")
print(df_final['priority'].value_counts())

print(f"\n Decision vs Priority Crosstab:")
print(pd.crosstab(df_final['decision'], df_final['priority'], margins=True))

print(f"\n Most Common Weaknesses:")
weakness_list = []
for weaknesses in df_final[df_final['weakness_flags'] != 'NONE']['weakness_flags']:
    weakness_list.extend(weaknesses.split(','))
weakness_df = pd.Series(weakness_list).value_counts()
print(weakness_df)

print(f"\n Deadline Summary:")
print(f"  - Items dengan deadline: {df_final['deadline'].notna().sum()}")
print(f"  - Evaluation deadline (14 hari): {(df_final['decision'] == 'EVALUASI').sum()}")
print(f"  - Suspension review (30 hari): {(df_final['decision'] == 'TARIK').sum()}")

DECISION ENGINE - ANALYSIS REPORT

 Decision Summary:
decision
EVALUASI       10557
TARIK           2399
PERTAHANKAN     1992
Name: count, dtype: int64

 Priority Distribution:
priority
MEDIUM      9821
HIGH        2428
LOW         1992
CRITICAL     707
Name: count, dtype: int64

 Decision vs Priority Crosstab:
priority     CRITICAL  HIGH   LOW  MEDIUM    All
decision                                        
EVALUASI            0   736     0    9821  10557
PERTAHANKAN         0     0  1992       0   1992
TARIK             707  1692     0       0   2399
All               707  2428  1992    9821  14948

 Most Common Weaknesses:
HARGA         3299
KUALITAS      2763
LAYANAN       2524
PENGIRIMAN    2108
Name: count, dtype: int64

 Deadline Summary:
  - Items dengan deadline: 12956
  - Evaluation deadline (14 hari): 10557
  - Suspension review (30 hari): 2399


In [12]:
print(" STEP 5 COMPLETE - Decision Engine")
print(f"\nOutput file ready: product_decisions.csv")
print(f"Next step: Step 6 - Visualisasi & Report")

 STEP 5 COMPLETE - Decision Engine

Output file ready: product_decisions.csv
Next step: Step 6 - Visualisasi & Report
